# Make 2DAlphabet ROOT inputs

Reads coffea output files and writes `mtt_vs_mt` histograms into ROOT files
suitable for the 2DAlphabet fit framework, split by pass/fail and central/forward
analysis categories.

In [130]:
import sys
from pathlib import Path
import uproot
from coffea.util import load

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    p for p in [cwd, *cwd.parents]
    if (p / "python").is_dir() and (p / "outputs").is_dir()
)
if str(REPO_ROOT / "python") not in sys.path:
    sys.path.append(str(REPO_ROOT / "python"))

year = "2024"
coffea_dir = REPO_ROOT / "outputs" / "dy"
out_dir = REPO_ROOT / "outputs" / "2dalphabet_inputs"

## Shared helpers

In [131]:
# at = ≥1 top-tagged jet; 2t = ≥2; cen/fwd = barrel/forward
ANCAT = {"atcen": 0, "atfwd": 1, "2tcen": 2, "2tfwd": 3}

# histogram key names expected by 2DAlphabet
HIST_KEYS = {
    "pass_cen": f"MttvsMtCen{year}Pass",
    "fail_cen": f"MttvsMtCen{year}Fail",
    "pass_fwd": f"MttvsMtFwd{year}Pass",
    "fail_fwd": f"MttvsMtFwd{year}Fail",
}

## Data

In [132]:
data_paths = sorted(coffea_dir.glob(f"data_{year}_*_noSyst.coffea"))

data_total = None
for path in data_paths:
    h = load(path)["mtt_vs_mt"]
    data_total = h if data_total is None else data_total + h

In [133]:
sel = {"systematic": "nominal"}

data_pass_cen = data_total[{**sel, "anacat": ANCAT["2tcen"]}]
data_fail_cen = data_total[{**sel, "anacat": ANCAT["atcen"]}]
data_pass_fwd = data_total[{**sel, "anacat": ANCAT["2tfwd"]}]
data_fail_fwd = data_total[{**sel, "anacat": ANCAT["atfwd"]}]

In [134]:
data_out = out_dir / f"data_{year}.root"
with uproot.recreate(data_out) as f:
    f[HIST_KEYS["pass_cen"]] = data_pass_cen
    f[HIST_KEYS["fail_cen"]] = data_fail_cen
    f[HIST_KEYS["pass_fwd"]] = data_pass_fwd
    f[HIST_KEYS["fail_fwd"]] = data_fail_fwd

print(f"Wrote {data_out}")

Wrote /home/cms-jovyan/new_git/TTbarHadronicSkimmer_coffea2025/outputs/2dalphabet_inputs/data_2024.root


## TTbar (MC)

In [135]:
ttbar_paths = list(coffea_dir.glob(f"TTbar_{year}_inclusive_noSyst.coffea"))
h_ttbar = load(ttbar_paths[0])["mtt_vs_mt"]

In [136]:
sel = {"systematic": "nominal"}

ttbar_pass_cen = h_ttbar[{**sel, "anacat": ANCAT["2tcen"]}]
ttbar_fail_cen = h_ttbar[{**sel, "anacat": ANCAT["atcen"]}]
ttbar_pass_fwd = h_ttbar[{**sel, "anacat": ANCAT["2tfwd"]}]
ttbar_fail_fwd = h_ttbar[{**sel, "anacat": ANCAT["atfwd"]}]

In [137]:
ttbar_out = out_dir / f"TTbar_{year}.root"
with uproot.recreate(ttbar_out) as f:
    f[HIST_KEYS["pass_cen"]] = ttbar_pass_cen
    f[HIST_KEYS["fail_cen"]] = ttbar_fail_cen
    f[HIST_KEYS["pass_fwd"]] = ttbar_pass_fwd
    f[HIST_KEYS["fail_fwd"]] = ttbar_fail_fwd

print(f"Wrote {ttbar_out}")

Wrote /home/cms-jovyan/new_git/TTbarHadronicSkimmer_coffea2025/outputs/2dalphabet_inputs/TTbar_2024.root


## Save histograms

In [138]:
import pickle

hists = {
    "data": data_total,
    "ttbar": h_ttbar,
}

pkl_out = out_dir / f"hists_{year}.pkl"
with open(pkl_out, "wb") as f:
    pickle.dump(hists, f)

print(f"Wrote {pkl_out}")

Wrote /home/cms-jovyan/new_git/TTbarHadronicSkimmer_coffea2025/outputs/2dalphabet_inputs/hists_2024.pkl
